# Análise inicial do `FaceGest_landmarks.csv`

## Resumo geral

Este notebook apresenta uma análise exploratória do conjunto de landmarks faciais. Cada registro válido é interpretado como:

- um rótulo numérico de classe;
- 468 landmarks do MediaPipe Face Mesh;
- três coordenadas por landmark (`x`, `y` e `z`);
- 1.405 valores por linha: `1 + 468 × 3`.

A análise verifica a estrutura e a integridade dos registros, a distribuição das classes, os limites das coordenadas, a variabilidade dos landmarks, diferenças geométricas entre classes e a ordem temporal da coleta.

Os nomes dos gestos são associados aos rótulos conforme o mapeamento documentado em `data/infos.md`. A análise de distribuição ordena essas classes pelo número de amostras e identifica as três mais frequentes.

## 1. Preparação do ambiente

A análise utiliza Pandas para a leitura tabular, NumPy para os cálculos vetorizados e Matplotlib para as visualizações. Como o CSV possui vários gigabytes, `pandas.read_csv` é utilizado com `chunksize`, processando um bloco por vez e mantendo o uso de memória controlado.

In [22]:
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 110})

### Configuração de exibição do Pandas

As opções abaixo deixam as tabelas compactas e exibem os valores numéricos com precisão suficiente para esta análise exploratória.

In [23]:
pd.options.display.max_columns = 20
pd.options.display.max_rows = 30
pd.options.display.float_format = "{:.6f}".format

### Localização do arquivo e parâmetros

O caminho é resolvido tanto para notebooks iniciados na raiz do projeto quanto dentro da pasta `notebooks`. As amostras mantidas em memória usam *reservoir sampling*, garantindo seleção uniforme e reproduzível sem armazenar todo o dataset.

In [24]:
candidate_paths = [
    Path("../data/FaceGest_landmarks.csv"),
    Path("data/FaceGest_landmarks.csv"),
]
CSV_PATH = next((path.resolve() for path in candidate_paths if path.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError("Não foi possível localizar data/FaceGest_landmarks.csv.")

N_LANDMARKS = 468
AXES = ("x", "y", "z")
EXPECTED_COLUMNS = 1 + N_LANDMARKS * len(AXES)
SEED = 42
GLOBAL_SAMPLE_ROWS = 800
PER_CLASS_SAMPLE_ROWS = 400
SEQUENCE_STEP = 25
CHUNK_SIZE = 2_500
TOP_N_CLASSES = 3
CLASS_NAMES = {
    0: "Single blink right",
    1: "Single blink left",
    2: "Double blink",
    3: "Raise eyebrows",
    4: "Mouth open",
    5: "Smile",
    6: "Frown",
    7: "Lips pursed",
    8: "Nodding (up and down)",
    9: "Shake head (horizontally)",
    10: "Blink + Smile",
    11: "Raise eyebrows + Mouth open",
    12: "Wink + Head tilt",
}
COLUMN_NAMES = ["label"] + [
    f"landmark_{landmark}_{axis}"
    for landmark in range(N_LANDMARKS)
    for axis in AXES
]

def class_name(label):
    return CLASS_NAMES.get(label, f"Unknown label {label}")

configuration = pd.DataFrame([{
    "arquivo": CSV_PATH,
    "landmarks": N_LANDMARKS,
    "colunas_esperadas": EXPECTED_COLUMNS,
    "linhas_por_bloco": CHUNK_SIZE,
    "classes_a_selecionar": TOP_N_CLASSES,
    "amostra_global": GLOBAL_SAMPLE_ROWS,
    "amostra_por_classe": PER_CLASS_SAMPLE_ROWS,
}])
configuration

,arquivo,landmarks,colunas_esperadas,linhas_por_bloco,amostra_global,amostra_por_classe
0,/home/luizcarlos/projetos_ufpb/DinoGesture/dat...,468,1405,2500,800,400


## 2. Leitura incremental

Uma linha é considerada válida quando possui exatamente 1.405 valores, todos são finitos e o primeiro valor representa um rótulo inteiro.

A passagem pelo arquivo acumula várias estatísticas ao mesmo tempo para evitar releituras de vários gigabytes. A preparação dos acumuladores, a leitura e a consolidação dos resultados estão separadas em células diferentes.

### 2.1 Inicialização dos acumuladores

Os acumuladores armazenam contagens, somas, mínimos, máximos e pequenas amostras. Nenhum deles cresce proporcionalmente ao número total de coordenadas do arquivo, com exceção do conjunto compacto de assinaturas usado para detectar duplicatas exatas.

In [26]:
n_complete_lines = 0
n_valid = 0
n_malformed = 0
n_nonfinite = 0
n_invalid_label = 0
n_exact_duplicates = 0

class_counts = Counter()
coordinate_sum = np.zeros((N_LANDMARKS, 3), dtype=np.float64)
coordinate_sumsq = np.zeros((N_LANDMARKS, 3), dtype=np.float64)
coordinate_min = np.full((N_LANDMARKS, 3), np.inf)
coordinate_max = np.full((N_LANDMARKS, 3), -np.inf)
xy_outside_unit = np.zeros(2, dtype=np.int64)

feature_names = (
    "centro_x", "centro_y", "centro_z",
    "dispersao_x", "dispersao_y", "dispersao_z",
    "amplitude_x", "amplitude_y", "amplitude_z",
)
class_feature_sum = defaultdict(lambda: np.zeros(len(feature_names)))
class_coordinate_sum = defaultdict(lambda: np.zeros((N_LANDMARKS, 3)))
class_feature_samples = defaultdict(list)
class_sample_seen = Counter()
representative_points = {}
global_points_sample = []
sequence_indices = []
sequence_labels = []
runs = []
seen_hashes = set()
rng = np.random.default_rng(SEED)

### 2.2 Processamento do dataset

Esta é a etapa mais demorada. A assinatura de cada linha válida permite contar duplicatas exatas sem guardar novamente todas as coordenadas. O Pandas percorre o CSV em blocos e os cálculos de cada bloco são vetorizados com NumPy.

In [ ]:
timer_start = time.perf_counter()
current_run_label = None
current_run_start = 0
current_run_length = 0

chunks = pd.read_csv(
    CSV_PATH,
    header=None,
    names=COLUMN_NAMES,
    dtype=np.float64,
    chunksize=CHUNK_SIZE,
    on_bad_lines="skip",
)

for chunk in chunks:
    n_complete_lines += len(chunk)
    values = chunk.to_numpy(dtype=np.float64, copy=False)

    missing_mask = np.isnan(values).any(axis=1)
    infinite_mask = np.isinf(values).any(axis=1)
    finite_mask = ~(missing_mask | infinite_mask)
    integer_label_mask = np.zeros(len(chunk), dtype=bool)
    integer_label_mask[finite_mask] = (
        values[finite_mask, 0] == np.rint(values[finite_mask, 0])
    )
    valid_mask = finite_mask & integer_label_mask

    n_malformed += int(missing_mask.sum())
    n_nonfinite += int((infinite_mask & ~missing_mask).sum())
    n_invalid_label += int((finite_mask & ~integer_label_mask).sum())

    if not valid_mask.any():
        continue

    valid_chunk = chunk.loc[valid_mask]
    valid_values = values[valid_mask]
    labels = valid_values[:, 0].astype(np.int64)
    points_batch = valid_values[:, 1:].reshape(-1, N_LANDMARKS, 3)
    centroids = points_batch.mean(axis=1)
    dispersions = points_batch.std(axis=1)
    amplitudes = np.ptp(points_batch, axis=1)
    features_batch = np.concatenate((centroids, dispersions, amplitudes), axis=1)
    row_hashes = pd.util.hash_pandas_object(valid_chunk, index=False).to_numpy(dtype=np.uint64)

    coordinate_sum += points_batch.sum(axis=0)
    coordinate_sumsq += np.square(points_batch).sum(axis=0)
    coordinate_min = np.minimum(coordinate_min, points_batch.min(axis=0))
    coordinate_max = np.maximum(coordinate_max, points_batch.max(axis=0))
    xy_outside_unit += (
        (points_batch[:, :, :2] < 0) | (points_batch[:, :, :2] > 1)
    ).sum(axis=(0, 1))

    for label in np.unique(labels):
        label_mask = labels == label
        label_count = int(label_mask.sum())
        class_counts[int(label)] += label_count
        class_feature_sum[int(label)] += features_batch[label_mask].sum(axis=0)
        class_coordinate_sum[int(label)] += points_batch[label_mask].sum(axis=0)

    for label, points, features, row_hash in zip(labels, points_batch, features_batch, row_hashes):
        label = int(label)
        row_hash = int(row_hash)
        representative_points.setdefault(label, points.copy())

        if row_hash in seen_hashes:
            n_exact_duplicates += 1
        else:
            seen_hashes.add(row_hash)

        class_sample_seen[label] += 1
        class_bucket = class_feature_samples[label]
        if len(class_bucket) < PER_CLASS_SAMPLE_ROWS:
            class_bucket.append(features.copy())
        else:
            replacement = int(rng.integers(class_sample_seen[label]))
            if replacement < PER_CLASS_SAMPLE_ROWS:
                class_bucket[replacement] = features.copy()

        if len(global_points_sample) < GLOBAL_SAMPLE_ROWS:
            global_points_sample.append(points.copy())
        else:
            replacement = int(rng.integers(n_valid + 1))
            if replacement < GLOBAL_SAMPLE_ROWS:
                global_points_sample[replacement] = points.copy()

        if n_valid % SEQUENCE_STEP == 0:
            sequence_indices.append(n_valid)
            sequence_labels.append(label)

        if current_run_label is None:
            current_run_label = label
            current_run_start = n_valid
            current_run_length = 1
        elif label == current_run_label:
            current_run_length += 1
        else:
            runs.append((current_run_label, current_run_start, current_run_length))
            current_run_label = label
            current_run_start = n_valid
            current_run_length = 1

        n_valid += 1

if current_run_label is not None:
    runs.append((current_run_label, current_run_start, current_run_length))

elapsed_seconds = time.perf_counter() - timer_start
finished_at = datetime.now().astimezone()

### 2.3 Consolidação das métricas

As somas acumuladas são convertidas em médias, variâncias e desvios-padrão. Esta célula também organiza os rótulos e as amostras que serão reutilizados nas próximas seções.

In [ ]:
if n_valid == 0:
    raise ValueError("Nenhuma linha válida foi encontrada no dataset.")

coordinate_mean = coordinate_sum / n_valid
coordinate_variance = np.maximum(coordinate_sumsq / n_valid - coordinate_mean**2, 0)
coordinate_std = np.sqrt(coordinate_variance)
points_sample = np.stack(global_points_sample)
labels_sorted = sorted(class_counts)
class_labels = [class_name(label) for label in labels_sorted]
counts_array = np.array([class_counts[label] for label in labels_sorted])
class_percentages = 100 * counts_array / n_valid
imbalance_ratio = counts_array.max() / counts_array.min()
duplicate_rate = 100 * n_exact_duplicates / n_valid
run_lengths = np.array([length for _, _, length in runs])

## 3. Resumo do dataset

A tabela apresenta os principais indicadores do conjunto. A razão de desbalanceamento compara a maior classe com a menor; valores próximos de 1 indicam distribuição mais uniforme.

In [ ]:
summary_rows = [
    {"indicador": "Registros válidos", "resultado": f"{n_valid:,}"},
    {"indicador": "Classes encontradas", "resultado": f"{len(labels_sorted)}: {', '.join(map(str, labels_sorted))}"},
    {"indicador": "Validade das linhas completas", "resultado": f"{100 * n_valid / max(n_complete_lines, 1):.3f}%"},
    {"indicador": "Razão maior/menor classe", "resultado": f"{imbalance_ratio:.2f}×"},
    {"indicador": "Duplicatas exatas", "resultado": f"{n_exact_duplicates:,} ({duplicate_rate:.3f}%)"},
    {"indicador": "Blocos consecutivos de classe", "resultado": f"{len(runs):,}"},
    {"indicador": "Tempo de leitura", "resultado": f"{elapsed_seconds:.1f} s"},
    {"indicador": "Fim da análise", "resultado": finished_at.strftime("%Y-%m-%d %H:%M:%S %Z")},
]
summary = pd.DataFrame(summary_rows).rename(columns={
    "indicador": "Indicador",
    "resultado": "Resultado",
})
summary

## 4. Estrutura e integridade

Como o arquivo não possui cabeçalho, a posição define o conteúdo:

| Posição | Conteúdo |
|---|---|
| 0 | classe |
| 1, 2, 3 | landmark 0: `x`, `y`, `z` |
| 4, 5, 6 | landmark 1: `x`, `y`, `z` |
| ... | ... |
| 1.402, 1.403, 1.404 | landmark 467: `x`, `y`, `z` |

Linhas malformadas, valores não finitos e rótulos não inteiros representam problemas de conteúdo e devem ser revisados antes do treinamento.

In [ ]:
quality_rows = [
    {"tipo": "Válidas", "quantidade": f"{n_valid:,}"},
    {"tipo": "Malformadas", "quantidade": f"{n_malformed:,}"},
    {"tipo": "Com valores não finitos", "quantidade": f"{n_nonfinite:,}"},
    {"tipo": "Com rótulo inválido", "quantidade": f"{n_invalid_label:,}"},
    {"tipo": "Duplicatas exatas", "quantidade": f"{n_exact_duplicates:,}"},
]
quality = pd.DataFrame(quality_rows).rename(columns={
    "tipo": "Tipo",
    "quantidade": "Quantidade",
})
quality

In [ ]:
quality_labels = ["válidas", "malformadas", "não finitas", "rótulo inválido"]
quality_values = [n_valid, n_malformed, n_nonfinite, n_invalid_label]
colors = ["#2a9d8f", "#e76f51", "#e9c46a", "#f4a261"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(quality_labels, quality_values, color=colors)
ax.set_title("Integridade das linhas do dataset")
ax.set_ylabel("Quantidade de linhas")
ax.bar_label(bars, labels=[f"{value:,}" for value in quality_values], padding=3)
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 5. Distribuição das classes

A tabela relaciona cada rótulo numérico ao gesto definido em `data/infos.md`, ordena as classes da maior para a menor quantidade de amostras e marca as três que serão mantidas. A seleção utiliza exclusivamente a contagem de registros válidos de cada rótulo.

In [ ]:
class_distribution = pd.DataFrame({
    "Label": labels_sorted,
    "Gesto facial": [class_name(label) for label in labels_sorted],
    "Amostras": counts_array.astype(int),
    "Participação (%)": class_percentages,
})
class_distribution = class_distribution.sort_values(
    "Amostras", ascending=False
).reset_index(drop=True)
class_distribution.insert(0, "Ranking", np.arange(1, len(class_distribution) + 1))
class_distribution["Selecionada no top 3"] = np.where(
    class_distribution["Ranking"] <= TOP_N_CLASSES, "Sim", "Não"
)

top_class_labels = class_distribution.head(TOP_N_CLASSES)["Label"].astype(int).tolist()
top_class_names = class_distribution.head(TOP_N_CLASSES)["Gesto facial"].tolist()
class_distribution

In [ ]:
plot_names = class_distribution["Gesto facial"]
plot_counts = class_distribution["Amostras"]
bar_colors = np.where(
    class_distribution["Selecionada no top 3"] == "Sim", "#2a9d8f", "#adb5bd"
)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(plot_names, plot_counts, color=bar_colors)
ax.set(
    title="Ranking dos gestos por número de amostras — top 3 em destaque",
    xlabel="Gesto facial",
    ylabel="Amostras",
)
ax.bar_label(
    bars,
    labels=[f"{value:,}" for value in plot_counts],
    padding=3,
    fontsize=8,
)
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### 5.1 Gestos selecionados

A tabela abaixo contém somente os três gestos com maior número de amostras. A lista `top_class_labels` guarda os rótulos correspondentes e poderá ser reutilizada posteriormente para filtrar o dataset.

In [ ]:
top3_classes = class_distribution.head(TOP_N_CLASSES).copy()
top3_classes

### 5.2 Resultado e impacto da seleção

No CSV analisado, os três gestos mais frequentes são:

1. **Label 12 — Wink + Head tilt:** 27.074 amostras;
2. **Label 8 — Nodding (up and down):** 26.512 amostras;
3. **Label 2 — Double blink:** 24.725 amostras.

Juntas, essas classes possuem 78.311 das 225.528 amostras válidas, equivalentes a aproximadamente 34,72% do dataset. A tabela calculada abaixo também apresenta o total e o percentual que serão descartados ao conservar apenas o top 3.

In [ ]:
selected_samples = int(top3_classes["Amostras"].sum())
discarded_samples = int(n_valid - selected_samples)
selection_impact = pd.DataFrame({
    "Grupo": ["Top 3 mantido", "Demais classes descartadas"],
    "Amostras": [selected_samples, discarded_samples],
    "Percentual (%)": [
        100 * selected_samples / n_valid,
        100 * discarded_samples / n_valid,
    ],
})
selection_impact

## 6. Distribuição e limites das coordenadas

No MediaPipe, `x` e `y` são normalmente normalizados pelas dimensões da imagem. Valores fora de `[0, 1]` podem ocorrer quando partes da face extrapolam o enquadramento, mas devem ser inspecionados. O eixo `z` representa profundidade relativa e não deve ser validado com essa mesma faixa.

In [ ]:
axis_mean = coordinate_mean.mean(axis=0)
axis_min = coordinate_min.min(axis=0)
axis_max = coordinate_max.max(axis=0)
axis_std = np.sqrt(np.maximum(
    coordinate_sumsq.sum(axis=0) / (n_valid * N_LANDMARKS)
    - (coordinate_sum.sum(axis=0) / (n_valid * N_LANDMARKS)) ** 2,
    0,
))

axis_rows = [
    {
        "eixo": axis,
        "minimo": f"{axis_min[index]:.5f}",
        "media": f"{axis_mean[index]:.5f}",
        "desvio": f"{axis_std[index]:.5f}",
        "maximo": f"{axis_max[index]:.5f}",
        "fora_intervalo": f"{xy_outside_unit[index]:,}" if index < 2 else "não aplicável",
    }
    for index, axis in enumerate(AXES)
]
axis_statistics = pd.DataFrame(axis_rows).rename(columns={
    "eixo": "Eixo",
    "minimo": "Mínimo",
    "media": "Média",
    "desvio": "Desvio-padrão",
    "maximo": "Máximo",
    "fora_intervalo": "Fora de [0,1]",
})
axis_statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis_index, (axis, ax) in enumerate(zip(AXES, axes)):
    sampled_values = points_sample[:, :, axis_index].ravel()
    ax.hist(sampled_values, bins=60, color="#457b9d", alpha=0.85)
    ax.axvline(sampled_values.mean(), color="#d62828", linestyle="--", label="média da amostra")
    ax.set(title=f"Distribuição de {axis}", xlabel=axis, ylabel="Frequência")
    ax.legend(fontsize=8)
fig.suptitle(f"Distribuições em amostra uniforme de {len(points_sample):,} registros", y=1.03)
plt.tight_layout()
plt.show()

## 7. Variabilidade espacial dos landmarks

O desvio-padrão de cada landmark ajuda a localizar as regiões que mais mudam. Pontos muito variáveis podem carregar informação do gesto, mas também podem refletir movimento global da cabeça, distância da câmera ou instabilidade de detecção. A variabilidade combinada é a norma dos desvios nos três eixos.

In [ ]:
landmark_variability = np.linalg.norm(coordinate_std, axis=1)
top_indices = np.argsort(landmark_variability)[-15:][::-1]
top_landmark_rows = [
    {
        "posicao": rank,
        "landmark": index,
        "variabilidade": f"{landmark_variability[index]:.6f}",
        "std_x": f"{coordinate_std[index, 0]:.6f}",
        "std_y": f"{coordinate_std[index, 1]:.6f}",
        "std_z": f"{coordinate_std[index, 2]:.6f}",
    }
    for rank, index in enumerate(top_indices, start=1)
]
top_landmarks = pd.DataFrame(top_landmark_rows).rename(columns={
    "posicao": "Posição",
    "landmark": "Landmark",
    "variabilidade": "Variabilidade combinada",
    "std_x": "std(x)",
    "std_y": "std(y)",
    "std_z": "std(z)",
})
top_landmarks

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for axis_index, axis in enumerate(AXES):
    ax.plot(coordinate_std[:, axis_index], label=f"std({axis})", alpha=0.85)
ax.scatter(top_indices, landmark_variability[top_indices], color="black", s=18, label="top 15 (norma xyz)")
ax.set(title="Variabilidade por índice de landmark", xlabel="Índice do landmark", ylabel="Desvio-padrão")
ax.legend(ncol=4)
plt.tight_layout()
plt.show()

## 8. Exemplos da geometria facial

O primeiro registro válido de cada classe é usado como exemplo. Esta verificação visual ajuda a detectar faces deslocadas, escalas muito diferentes ou landmarks anômalos. A cor representa a profundidade `z`; o eixo `y` é invertido para acompanhar a orientação da imagem.

In [ ]:
classes_to_show = labels_sorted[:12]
ncols = min(4, len(classes_to_show))
nrows = math.ceil(len(classes_to_show) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows), squeeze=False)

for ax, label in zip(axes.ravel(), classes_to_show):
    example_points = representative_points[label]
    ax.scatter(example_points[:, 0], example_points[:, 1], c=example_points[:, 2], s=7, cmap="coolwarm")
    ax.set_title(f"Classe {class_name(label)}")
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()
    ax.set_xlabel("x")
    ax.set_ylabel("y")

for ax in axes.ravel()[len(classes_to_show):]:
    ax.axis("off")

fig.suptitle("Primeiro registro válido de cada classe", y=1.01)
plt.tight_layout()
plt.show()

## 9. Características geométricas por classe

Cada face é resumida por centro, dispersão e amplitude em cada eixo. O mapa de calor padroniza as médias entre as classes: valores positivos estão acima da média daquela característica e valores negativos estão abaixo. Diferenças fortes no centro ou na escala podem indicar que um modelo aprenderia enquadramento e distância da câmera em vez do gesto.

In [ ]:
feature_means = np.vstack([class_feature_sum[label] / class_counts[label] for label in labels_sorted])
feature_scale = feature_means.std(axis=0)
feature_zscore = (feature_means - feature_means.mean(axis=0)) / np.where(feature_scale == 0, 1, feature_scale)

fig, ax = plt.subplots(figsize=(12, max(4, 0.55 * len(labels_sorted))))
image = ax.imshow(feature_zscore, cmap="RdBu_r", aspect="auto", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(len(feature_names)), feature_names, rotation=40, ha="right")
ax.set_yticks(range(len(labels_sorted)), class_labels)
ax.set_xlabel("Característica geométrica")
ax.set_ylabel("Classe")
ax.set_title("Perfil geométrico médio por classe")
fig.colorbar(image, ax=ax, label="z-score")
plt.tight_layout()
plt.show()

In [ ]:
sample_arrays = {label: np.vstack(class_feature_samples[label]) for label in labels_sorted}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for feature_index, title, ax in [
    (6, "Amplitude horizontal da face", axes[0]),
    (7, "Amplitude vertical da face", axes[1]),
]:
    boxplot_data = [sample_arrays[label][:, feature_index] for label in labels_sorted]
    ax.boxplot(boxplot_data, tick_labels=class_labels, showfliers=False)
    ax.set(title=title, xlabel="Classe", ylabel=feature_names[feature_index])

fig.suptitle("Escala da face por classe")
plt.tight_layout()
plt.show()

## 10. Ordem de coleta e risco de vazamento temporal

Frames vizinhos de um vídeo tendem a ser muito parecidos. Se as linhas forem divididas aleatoriamente, frames quase idênticos podem aparecer no treino e no teste, produzindo uma avaliação otimista. A quantidade e o tamanho dos blocos consecutivos indicam quanto a coleta está organizada por sequências de uma mesma classe.

In [ ]:
temporal_rows = [
    {"indicador": "Blocos consecutivos", "resultado": f"{len(runs):,}"},
    {"indicador": "Trocas de classe", "resultado": f"{max(len(runs) - 1, 0):,}"},
    {"indicador": "Tamanho mediano do bloco", "resultado": f"{np.median(run_lengths):,.0f}"},
    {"indicador": "Maior bloco", "resultado": f"{run_lengths.max():,}"},
]
temporal_statistics = pd.DataFrame(temporal_rows).rename(columns={
    "indicador": "Indicador temporal",
    "resultado": "Resultado",
})
temporal_statistics

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.step(sequence_indices, sequence_labels, where="post", linewidth=1.2, color="#264653")
ax.set_yticks(labels_sorted, class_labels)
ax.set(
    title=f"Classe ao longo do arquivo: um ponto a cada {SEQUENCE_STEP} registros válidos",
    xlabel="Índice do registro válido",
    ylabel="Classe",
)
plt.tight_layout()
plt.show()

## 11. Conclusões e próximos passos

A interpretação dos resultados deve considerar os seguintes pontos:

- a taxa de linhas válidas deve permanecer próxima de 100%; linhas completas malformadas indicam um problema de exportação;
- duplicatas exatas podem aumentar artificialmente o volume e devem ser avaliadas antes do treinamento;
- o ranking por quantidade define os labels 12, 8 e 2 como as três classes selecionadas;
- após a seleção, o balanceamento deve ser recalculado considerando somente essas três classes;
- coordenadas `x` ou `y` fora de `[0, 1]` devem ser relacionadas ao enquadramento antes de qualquer descarte;
- diferenças de centro e escala entre classes podem criar atalhos indesejados para o modelo;
- poucos blocos consecutivos e muito longos aumentam o risco de vazamento quando a divisão é feita por frame.

### Próximos passos recomendados

1. Utilizar `top_class_labels` para filtrar o dataset pelos labels 12, 8 e 2.
2. Recalcular as estatísticas e o balanceamento após o filtro.
3. Separar treino, validação e teste por pessoa, sessão ou bloco temporal, e não por linhas aleatórias.
4. Normalizar translação, escala e, se necessário, rotação da face.
5. Avaliar o modelo com métricas por classe, especialmente precision, recall e F1.